# EU XREF Data Exploration

Goal: understand the content, quality, and vector-search value of the EU XREF views before building ground-truth match datasets.

This notebook focuses on table-level investigations, active/valid part filtering, description coverage, and XREF relationship structure.

## Setup

Expected `.env` keys:

- `GXR_EU_DB_HOST`
- `GXR_EU_DB_PORT`
- `GXR_EU_DB_SERVICE_NAME`
- `GXR_EU_DB_USER`
- `GXR_EU_DB_PASS`

This notebook uses `python-oracledb` first, then falls back to the same Oracle JDBC driver shape that works in DBeaver. On this workstation it can usually auto-discover DBeaver's bundled JRE and cached `ojdbc` jar. Optional `.env` overrides are `GXR_EU_JDBC_URL`, `OJDBC_JAR`, `JVM_DLL`, and `ORACLE_CLIENT_LIB_DIR`.

In [ ]:
import sys
print(sys.version)

In [ ]:
import sys
print(sys.version)
print(sys.executable)

In [ ]:
from pathlib import Path
print(Path.cwd())
print((Path.cwd() / ".env").exists())

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

keys = [
    "GXR_EU_DB_HOST",
    "GXR_EU_DB_PORT",
    "GXR_EU_DB_SERVICE_NAME",
    "GXR_EU_DB_USER",
    "GXR_EU_DB_PASS",
]

for key in keys:
    print(key, "SET" if os.getenv(key) else "MISSING")

In [ ]:
%pip install oracledb python-dotenv pandas matplotlib JPype1


In [ ]:
from pathlib import Path
from decimal import Decimal
import glob
import os
import urllib.request

import oracledb
import pandas as pd
from dotenv import load_dotenv

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / ".env").exists() and (path / "docs").exists():
            return path
    return start


ROOT = find_repo_root(Path.cwd().resolve())
load_dotenv(ROOT / ".env")

EU_ENV_KEYS = [
    "GXR_EU_DB_HOST",
    "GXR_EU_DB_PORT",
    "GXR_EU_DB_SERVICE_NAME",
    "GXR_EU_DB_USER",
    "GXR_EU_DB_PASS",
]
missing = [key for key in EU_ENV_KEYS if not os.getenv(key)]
if missing:
    raise RuntimeError(f"Missing EU database env vars: {missing}")

oracle_client_lib_dir = os.getenv("ORACLE_CLIENT_LIB_DIR")
if oracle_client_lib_dir:
    try:
        oracledb.init_oracle_client(lib_dir=oracle_client_lib_dir)
        print(f"Oracle thick mode initialized from {oracle_client_lib_dir}")
    except Exception as exc:
        raise RuntimeError(f"Failed to initialize Oracle thick mode from ORACLE_CLIENT_LIB_DIR={oracle_client_lib_dir!r}") from exc

dsn = f"//{os.getenv('GXR_EU_DB_HOST')}:{os.getenv('GXR_EU_DB_PORT', '1521')}/{os.getenv('GXR_EU_DB_SERVICE_NAME')}"
jdbc_url = os.getenv("GXR_EU_JDBC_URL") or f"jdbc:oracle:thin:@{dsn}"


def _first_existing(patterns: list[str]) -> str | None:
    for pattern in patterns:
        expanded = os.path.expandvars(os.path.expanduser(pattern))
        matches = sorted(glob.glob(expanded, recursive=True), reverse=True)
        if matches:
            return matches[0]
    return None


def _discover_ojdbc_jar() -> str | None:
    configured = os.getenv("OJDBC_JAR")
    if configured and Path(configured).exists():
        return configured

    cached = ROOT / ".cache" / "ojdbc11.jar"
    if cached.exists():
        return str(cached)
    return None


def _download_ojdbc_jar() -> str:
    cached = ROOT / ".cache" / "ojdbc11.jar"
    cached.parent.mkdir(parents=True, exist_ok=True)
    url = os.getenv("OJDBC_JAR_URL") or "https://repo1.maven.org/maven2/com/oracle/database/jdbc/ojdbc11/23.5.0.24.07/ojdbc11-23.5.0.24.07.jar"
    try:
        urllib.request.urlretrieve(url, cached)
        print(f"Downloaded JDBC driver to {cached}")
        return str(cached)
    except Exception as exc:
        raise RuntimeError(f"Unable to download Oracle JDBC jar from {url}: {exc}") from exc


def _discover_jvm_dll() -> str | None:
    configured = os.getenv("JVM_DLL")
    if configured and Path(configured).exists():
        return configured
    return _first_existing([
        r"%LOCALAPPDATA%\DBeaver\jre\bin\server\jvm.dll",
        r"%JAVA_HOME%\bin\server\jvm.dll",
        r"C:\Program Files\Java\**\bin\server\jvm.dll",
        r"~/Library/Java/JavaVirtualMachines/**/Contents/Home/lib/server/libjvm.dylib",
        r"/Library/Java/JavaVirtualMachines/**/Contents/Home/lib/server/libjvm.dylib",
        r"/Applications/DBeaver.app/Contents/Eclipse/jre/Contents/Home/lib/server/libjvm.dylib",
        r"/Applications/DBeaver.app/Contents/Eclipse/jre/Contents/Home/lib/server/libjvm.dylib",
        r"/opt/homebrew/opt/openjdk*/lib/server/libjvm.dylib",
        r"/opt/homebrew/Cellar/openjdk*/libexec/openjdk.jdk/Contents/Home/lib/server/libjvm.dylib",
        r"/usr/lib/jvm/**/lib/server/libjvm.so",
    ])


def connect_oracle():
    return oracledb.connect(
        user=os.getenv("GXR_EU_DB_USER"),
        password=os.getenv("GXR_EU_DB_PASS"),
        dsn=dsn,
    )


def connect_jdbc():
    import jpype

    ojdbc_jar = _discover_ojdbc_jar() or _download_ojdbc_jar()
    if not jpype.isJVMStarted():
        jvm_dll = _discover_jvm_dll()
        if not jvm_dll:
            raise RuntimeError("No JVM found. Set JVM_DLL in .env or install Java / DBeaver bundled JRE.")
        jpype.startJVM(jvm_dll, classpath=[ojdbc_jar], convertStrings=True)

    OracleDriver = jpype.JClass("oracle.jdbc.OracleDriver")
    DriverManager = jpype.JClass("java.sql.DriverManager")
    Properties = jpype.JClass("java.util.Properties")
    DriverManager.registerDriver(OracleDriver())
    props = Properties()
    props.setProperty("user", os.getenv("GXR_EU_DB_USER"))
    props.setProperty("password", os.getenv("GXR_EU_DB_PASS"))
    return DriverManager.getConnection(jdbc_url, props)


def _java_value_to_python(value):
    if value is None:
        return None
    class_name = value.getClass().getName() if hasattr(value, "getClass") else ""
    if class_name == "java.math.BigDecimal":
        text = str(value.toPlainString())
        decimal_value = Decimal(text)
        return int(decimal_value) if decimal_value == decimal_value.to_integral_value() else float(decimal_value)
    if class_name in {"java.sql.Timestamp", "java.sql.Date", "java.sql.Time"}:
        return str(value)
    return value


def jdbc_query_df(connection, sql: str, params: dict | None = None) -> pd.DataFrame:
    if params:
        raise NotImplementedError("JDBC fallback currently expects literal SQL. Keep notebook exploration queries parameter-free.")
    statement = connection.createStatement()
    result_set = None
    try:
        result_set = statement.executeQuery(sql)
        metadata = result_set.getMetaData()
        column_count = metadata.getColumnCount()
        columns = [metadata.getColumnLabel(index) for index in range(1, column_count + 1)]
        rows = []
        while result_set.next():
            rows.append([_java_value_to_python(result_set.getObject(index)) for index in range(1, column_count + 1)])
        return pd.DataFrame(rows, columns=columns)
    finally:
        if result_set is not None:
            result_set.close()
        statement.close()


DB_BACKEND = "auto"


def connect():
    global DB_BACKEND
    if DB_BACKEND == "jdbc":
        return connect_jdbc()
    if DB_BACKEND == "oracledb":
        return connect_oracle()
    try:
        connection = connect_oracle()
        DB_BACKEND = "oracledb"
        return connection
    except oracledb.NotSupportedError as exc:
        if "DPY-3015" not in str(exc):
            raise
        print("python-oracledb thin mode hit DPY-3015; falling back to JDBC.")
        DB_BACKEND = "jdbc"
        return connect_jdbc()


def query_df(sql: str, params: dict | None = None) -> pd.DataFrame:
    connection = connect()
    try:
        if DB_BACKEND == "jdbc":
            df = jdbc_query_df(connection, sql, params=params)
        else:
            df = pd.read_sql(sql, connection, params=params or {})
        df.columns = [str(column).lower() for column in df.columns]
        return df
    finally:
        connection.close()


OUTPUT_DIR = ROOT / "exports" / "eu_xref_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("EU Oracle config loaded:", {key: bool(os.getenv(key)) for key in EU_ENV_KEYS})
print("Oracle DSN:", dsn)
print("JDBC URL:", jdbc_url)
print("oracledb thin mode:", oracledb.is_thin_mode())
print("Discovered OJDBC jar:", _discover_ojdbc_jar() or _download_ojdbc_jar())
print("Discovered JVM:", _discover_jvm_dll())


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")

def save_df(df: pd.DataFrame, name: str) -> None:
    df.to_csv(OUTPUT_DIR / name, index=False)

def add_rate(df: pd.DataFrame, numerator: str, denominator: str, rate_name: str) -> pd.DataFrame:
    out = df.copy()
    out[rate_name] = (out[numerator] / out[denominator]).fillna(0)
    return out

def format_axis(ax, title: str, xlabel: str = "", ylabel: str = ""):
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.tight_layout()
    return ax

## Active / Valid Part Filter

The current vector-search active-part filter is used consistently in this notebook:

```sql
(product_type IS NULL OR product_type NOT IN ('04', '07'))
AND (discontinued_code IS NULL OR discontinued_code NOT IN ('M', 'O'))
AND restriction_code = 'RTSN'
```

For XREF relationship rows, the same logic is applied to the relevant `IP_` and/or `FP_` status columns.

## XREF_FISH_PART_LOCAL_EU_VW

One status row for each part in each country. This view tells us which internal parts are active/valid by country.

In [ ]:
local_country_active = query_df("""
    SELECT
        country_id,
        COUNT(DISTINCT part_number) AS distinct_parts,
        COUNT(DISTINCT CASE
            WHEN (product_type IS NULL OR product_type NOT IN ('04', '07'))
             AND (discontinued_code IS NULL OR discontinued_code NOT IN ('M', 'O'))
             AND restriction_code = 'RTSN'
            THEN part_number
        END) AS active_parts
    FROM XREF_ADMIN.XREF_FISH_PART_LOCAL_EU_VW
    GROUP BY country_id
    ORDER BY country_id
""")

local_country_active = add_rate(local_country_active, "active_parts", "distinct_parts", "active_part_rate")
save_df(local_country_active, "xref_fish_part_local_country_active.csv")
display(local_country_active)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

local_country_active.plot.bar(x="country_id", y=["distinct_parts", "active_parts"], ax=axes[0])
format_axis(axes[0], "Distinct vs active parts by country", "Country", "Parts")

local_country_active.plot.bar(x="country_id", y="active_part_rate", ax=axes[1], legend=False, color="tab:green")
axes[1].set_ylim(0, 1)
format_axis(axes[1], "Active part rate by country", "Country", "Active / distinct")

plt.show()

In [ ]:
local_status_breakdown = query_df("""
    SELECT
        country_id,
        product_type,
        discontinued_code,
        restriction_code,
        COUNT(*) AS row_count,
        COUNT(DISTINCT part_number) AS distinct_parts
    FROM XREF_ADMIN.XREF_FISH_PART_LOCAL_EU_VW
    GROUP BY country_id, product_type, discontinued_code, restriction_code
    ORDER BY country_id, row_count DESC
""")

save_df(local_status_breakdown, "xref_fish_part_local_status_breakdown.csv")
display(local_status_breakdown.head(50))

## XREF_FISH_PART_DESCRIPTN_EU_VW

One row for each internal part and language where an ODS Part Master description is available. This is the first place to inspect multilingual internal-part text coverage.

In [ ]:
fish_description_language_stats = query_df("""
    SELECT
        language_code,
        COUNT(*) AS row_count,
        COUNT(DISTINCT part_number) AS distinct_parts,
        AVG(LENGTH(part_description)) AS avg_description_length,
        SUM(CASE WHEN REGEXP_LIKE(part_description, '[^ -~]') THEN 1 ELSE 0 END) AS non_ascii_description_rows
    FROM XREF_ADMIN.XREF_FISH_PART_DESCRIPTN_EU_VW
    GROUP BY language_code
    ORDER BY row_count DESC
""")

fish_description_language_stats = add_rate(
    fish_description_language_stats,
    "non_ascii_description_rows",
    "row_count",
    "non_ascii_row_rate",
)
save_df(fish_description_language_stats, "xref_fish_part_descriptn_language_stats.csv")
display(fish_description_language_stats)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fish_description_language_stats.plot.bar(x="language_code", y="row_count", ax=axes[0], legend=False)
format_axis(axes[0], "Description rows by language", "Language", "Rows")

fish_description_language_stats.plot.bar(x="language_code", y="avg_description_length", ax=axes[1], legend=False, color="tab:orange")
format_axis(axes[1], "Average description length", "Language", "Characters")

fish_description_language_stats.plot.bar(x="language_code", y="non_ascii_row_rate", ax=axes[2], legend=False, color="tab:purple")
format_axis(axes[2], "Non-ASCII description rate", "Language", "Rate")

plt.show()

### Questions

- For how many parts are there multiple language rows sharing identical descriptions?
- Which language pairs have the most identical-description overlap?
- How many parts only have a description for one language?

In [ ]:
fish_description_identical_language_overlap = query_df("""
    WITH cleaned AS (
        SELECT
            part_number,
            language_code,
            TRIM(part_description) AS dtext
        FROM XREF_ADMIN.XREF_FISH_PART_DESCRIPTN_EU_VW
        WHERE part_description IS NOT NULL
    )
    SELECT
        d1.language_code AS l1,
        d2.language_code AS l2,
        COUNT(*) AS row_pairs,
        COUNT(DISTINCT d1.part_number) AS parts
    FROM cleaned d1
    JOIN cleaned d2
      ON d1.part_number = d2.part_number
     AND d1.language_code < d2.language_code
     AND d1.dtext = d2.dtext
    GROUP BY d1.language_code, d2.language_code
    ORDER BY parts DESC
""")

save_df(fish_description_identical_language_overlap, "xref_fish_part_descriptn_identical_language_overlap.csv")
display(fish_description_identical_language_overlap)


In [ ]:
overlap_plot_df = fish_description_identical_language_overlap.assign(
    language_pair=lambda df: df["l1"] + "-" + df["l2"]
)

ax = overlap_plot_df.plot.bar(
    x="language_pair",
    y="parts",
    figsize=(10, 5),
    legend=False,
)
format_axis(ax, "Identical descriptions by language pair", "Language pair", "Distinct parts")
plt.show()


In [ ]:
fish_description_language_count_distribution = query_df("""
    WITH part_language_counts AS (
        SELECT
            part_number,
            COUNT(DISTINCT language_code) AS language_count
        FROM XREF_ADMIN.XREF_FISH_PART_DESCRIPTN_EU_VW
        WHERE part_description IS NOT NULL
        GROUP BY part_number
    )
    SELECT
        language_count,
        COUNT(*) AS distinct_parts
    FROM part_language_counts
    GROUP BY language_count
    ORDER BY language_count
""")

save_df(fish_description_language_count_distribution, "xref_fish_part_descriptn_language_count_distribution.csv")
display(fish_description_language_count_distribution)

In [ ]:
ax = fish_description_language_count_distribution.plot.bar(
    x="language_count",
    y="distinct_parts",
    figsize=(8, 5),
    legend=False,
    color="tab:green",
)
format_axis(ax, "Parts by available description-language count", "Languages available", "Distinct parts")
plt.show()

## XREF_FISH_PART_MASTER_EU_VW

Expected to represent internal part master data. It appears to have one row per part number in intent, but observed stats show duplicate part numbers. Descriptions are longer/different from `XREF_FISH_PART_DESCRIPTN_EU_VW`, so this view needs separate quality checks.

In [ ]:
fish_master_description_stats = query_df("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT part_number) AS distinct_parts,
        AVG(LENGTH(part_description)) AS avg_description_length,
        SUM(CASE WHEN REGEXP_LIKE(part_description, '[^ -~]') THEN 1 ELSE 0 END) AS non_ascii_description_rows
    FROM XREF_ADMIN.XREF_FISH_PART_MASTER_EU_VW
""")

fish_master_description_stats = add_rate(
    fish_master_description_stats,
    "non_ascii_description_rows",
    "row_count",
    "non_ascii_row_rate",
)
save_df(fish_master_description_stats, "xref_fish_part_master_description_stats.csv")
display(fish_master_description_stats)

### Questions

- There are roughly 500k non-distinct rows. Are those duplicate part rows carrying different descriptions?

In [ ]:
fish_master_duplicate_summary = query_df("""
    WITH per_part AS (
        SELECT
            part_number,
            COUNT(*) AS rows_per_part,
            COUNT(DISTINCT NVL(TRIM(part_description), '<<NULL>>')) AS distinct_descriptions_per_part,
            SUM(CASE WHEN part_description IS NOT NULL THEN 1 ELSE 0 END) AS non_null_description_rows
        FROM XREF_ADMIN.XREF_FISH_PART_MASTER_EU_VW
        GROUP BY part_number
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS duplicate_part_numbers,
        SUM(rows_per_part) AS duplicate_part_rows,
        SUM(CASE WHEN distinct_descriptions_per_part > 1 THEN 1 ELSE 0 END) AS duplicate_parts_with_different_descriptions,
        SUM(CASE WHEN distinct_descriptions_per_part = 1 THEN 1 ELSE 0 END) AS duplicate_parts_with_same_or_all_null_description
    FROM per_part
""")

save_df(fish_master_duplicate_summary, "xref_fish_part_master_duplicate_summary.csv")
display(fish_master_duplicate_summary)

In [ ]:
fish_master_duplicate_description_distribution = query_df("""
    WITH per_part AS (
        SELECT
            part_number,
            COUNT(*) AS rows_per_part,
            COUNT(DISTINCT NVL(TRIM(part_description), '<<NULL>>')) AS distinct_descriptions_per_part
        FROM XREF_ADMIN.XREF_FISH_PART_MASTER_EU_VW
        GROUP BY part_number
        HAVING COUNT(*) > 1
    )
    SELECT
        rows_per_part,
        distinct_descriptions_per_part,
        COUNT(*) AS part_count
    FROM per_part
    GROUP BY rows_per_part, distinct_descriptions_per_part
    ORDER BY rows_per_part, distinct_descriptions_per_part
""")

save_df(fish_master_duplicate_description_distribution, "xref_fish_part_master_duplicate_description_distribution.csv")
display(fish_master_duplicate_description_distribution.head(50))

In [ ]:
plot_df = fish_master_duplicate_description_distribution.groupby("distinct_descriptions_per_part", as_index=False)["part_count"].sum()
ax = plot_df.plot.bar(
    x="distinct_descriptions_per_part",
    y="part_count",
    figsize=(8, 5),
    legend=False,
    color="tab:red",
)
format_axis(ax, "Duplicate parts by distinct-description count", "Distinct descriptions per duplicate part", "Part count")
plt.show()

## XREF_COMP_VEND_FISH_EU_VW

One row for each XREF relationship between two parts.

- `COMP2FISH`: `IP_` columns are external/input competitor parts; `FP_` columns are internal/output Fisher parts.
- `FISH2FISH`: both `IP_` and `FP_` columns are internal Fisher parts.
- Only `COMP2FISH` and `FISH2FISH` are relevant for the first ground-truth vector-search datasets.
- Active filters are applied to relevant `IP_` and `FP_` status columns for investigational counts.

In [ ]:
xref_relationship_breakdown = query_df("""
    SELECT
        cross_type,
        relationship_code_msg,
        COUNT(*) AS row_count,
        COUNT(DISTINCT ip_number) AS distinct_ip_numbers,
        COUNT(DISTINCT fp_number) AS distinct_fp_numbers
    FROM XREF_ADMIN.XREF_COMP_VEND_FISH_EU_VW
    GROUP BY cross_type, relationship_code_msg
    ORDER BY row_count DESC
""")

save_df(xref_relationship_breakdown, "xref_comp_vend_fish_relationship_breakdown.csv")
display(xref_relationship_breakdown)

In [ ]:
relationship_plot_df = xref_relationship_breakdown.assign(
    relationship=lambda df: df["cross_type"].fillna("NULL") + " | " + df["relationship_code_msg"].fillna("NULL")
).head(12)

ax = relationship_plot_df.sort_values("row_count").plot.barh(
    x="relationship",
    y="row_count",
    figsize=(10, 6),
    legend=False,
)
format_axis(ax, "Largest XREF relationship groups", "Rows", "Relationship")
plt.show()

In [ ]:
xref_relevant_active_counts = query_df("""
    SELECT
        cross_type,
        relationship_code_msg,
        COUNT(*) AS row_count,
        SUM(CASE
            WHEN (ip_product_type IS NULL OR ip_product_type NOT IN ('04', '07'))
             AND (ip_discontinued_code IS NULL OR ip_discontinued_code NOT IN ('M', 'O'))
            THEN 1 ELSE 0
        END) AS active_ip_rows,
        SUM(CASE
            WHEN (fp_product_type IS NULL OR fp_product_type NOT IN ('04', '07'))
             AND (fp_discontinued_code IS NULL OR fp_discontinued_code NOT IN ('M', 'O'))
            THEN 1 ELSE 0
        END) AS active_fp_rows,
        SUM(CASE
            WHEN (ip_product_type IS NULL OR ip_product_type NOT IN ('04', '07'))
             AND (ip_discontinued_code IS NULL OR ip_discontinued_code NOT IN ('M', 'O'))
             AND (fp_product_type IS NULL OR fp_product_type NOT IN ('04', '07'))
             AND (fp_discontinued_code IS NULL OR fp_discontinued_code NOT IN ('M', 'O'))
            THEN 1 ELSE 0
        END) AS active_ip_and_fp_rows
    FROM XREF_ADMIN.XREF_COMP_VEND_FISH_EU_VW
    WHERE cross_type IN ('COMP2FISH', 'FISH2FISH')
    GROUP BY cross_type, relationship_code_msg
    ORDER BY cross_type, row_count DESC
""")

xref_relevant_active_counts = add_rate(xref_relevant_active_counts, "active_ip_and_fp_rows", "row_count", "active_ip_and_fp_rate")
save_df(xref_relevant_active_counts, "xref_comp_vend_fish_relevant_active_counts.csv")
display(xref_relevant_active_counts)

In [ ]:
xref_relevant_examples = query_df("""
    SELECT
        cross_type,
        relationship_code_msg,
        fp_product_type,
        fp_discontinued_code,
        ip_number,
        ip_vendor_name,
        ip_vendor_part_number,
        fp_number,
        fp_vendor_name,
        fp_vendor_part_number,
        ip_description,
        fp_leaf_node_id
    FROM XREF_ADMIN.XREF_COMP_VEND_FISH_EU_VW
    WHERE cross_type IN ('COMP2FISH')
      AND relationship_code_msg IN ('Exact Match')
      AND (fp_product_type IS NULL OR fp_product_type NOT IN ('04', '07'))
      AND (fp_discontinued_code IS NULL OR fp_discontinued_code NOT IN ('M', 'O'))
      AND LENGTH(TRIM(ip_description)) > 0
""")

xref_relevant_examples.to_csv("xref_relevant_examples.csv", index=False)
print("CSV saved successfully.")

In [ ]:
import pandas as pd

df = pd.read_csv("xref_relevant_examples_sample_200k.csv")
df["description_length"] = df["ip_description"].fillna("").astype(str).str.len()
print(df.head())

In [ ]:
import pandas as pd

# -----------------------------
# 1. Load CSV
# -----------------------------

INPUT_CSV = "xref_relevant_examples_sample_200k.csv"

df = pd.read_csv(INPUT_CSV)

print("Number of rows:", len(df))
print("Columns:")
print(df.columns.tolist())


# -----------------------------
# 2. Compute description length
# -----------------------------

df["description_length"] = (
    df["ip_description"]
    .fillna("")
    .astype(str)
    .str.len()
)


# -----------------------------
# 3. Show descriptions + lengths
# -----------------------------

display(
    df[
        [
            "ip_description",
            "description_length"
        ]
    ].head(20)
)


# -----------------------------
# 4. Show description length stats
# -----------------------------

print("\nDescription length statistics:")

display(
    df["description_length"].describe()
)


# -----------------------------
# 5. Show useful percentiles
# -----------------------------

print("\nDescription length percentiles:")

display(
    df["description_length"].quantile(
        [
            0.00,
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00
        ]
    )
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.hist(
    df["description_length"],
    bins=50
)

plt.xlabel("Description Length (characters)")
plt.ylabel("Number of Rows")
plt.title("Distribution of Description Length")

plt.show()

In [ ]:
print("Minimum length:", df["description_length"].min())
print("Maximum length:", df["description_length"].max())
print("Mean length:", df["description_length"].mean())
print("Median length:", df["description_length"].median())

In [ ]:
import numpy as np
import pandas as pd

df["description_length"] = df["ip_description"].fillna("").astype(str).str.len()

bins = [
    0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100,
    120, 140, 160, 180, 200, 220, 240, 250, 254, 255
]

labels = [
    "0-9", "10-19", "20-29", "30-39", "40-49",
    "50-59", "60-69", "70-79", "80-89", "90-99",
    "100-119", "120-139", "140-159", "160-179",
    "180-199", "200-219", "220-239", "240-249",
    "250-253", "254"
]

df["length_bucket"] = pd.cut(
    df["description_length"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

display(df[["ip_description", "description_length", "length_bucket"]].head(20))

In [ ]:
df["length_bucket"] = pd.cut(
    df["description_length"],
    bins=bins,
    right=False,
    include_lowest=True
)

print(df["length_bucket"].cat.categories)

In [ ]:
joint_counts = (
    df.groupby(
        ["fp_leaf_node_id", "length_bucket"],
        observed=True
    )
    .size()
    .reset_index(name="population_count")
)

print("Number of joint groups:", len(joint_counts))
display(joint_counts.head(20))

In [ ]:
# ============================================================
# STEP 4: Calculate sample quota for each leaf node
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

# For CURRENT 200k test dataset:
TARGET_N = 200_000

# For the real ~2M dataset later, use:
# TARGET_N = 200_000


# ------------------------------------------------------------
# 1. Count population rows for every leaf
# ------------------------------------------------------------

leaf_counts = (
    df.groupby("fp_leaf_node_id")
      .size()
      .reset_index(name="leaf_population_count")
)

TOTAL_N = len(df)

print("Population rows:", TOTAL_N)
print("Target sample rows:", TARGET_N)
print("Unique leaf IDs:", len(leaf_counts))

display(leaf_counts.head(20))


# ------------------------------------------------------------
# 2. Calculate purely proportional leaf quota
# ------------------------------------------------------------

leaf_counts["ideal_leaf_quota"] = (
    TARGET_N
    * leaf_counts["leaf_population_count"]
    / TOTAL_N
)

display(
    leaf_counts.sort_values(
        "leaf_population_count"
    ).head(20)
)

In [ ]:
# ============================================================
# STEP 5: Protect rare leaf IDs
# ============================================================

# Start by guaranteeing 1 row per leaf
leaf_counts["base_quota"] = 1

# Obviously cannot sample more rows than exist
leaf_counts["base_quota"] = np.minimum(
    leaf_counts["base_quota"],
    leaf_counts["leaf_population_count"]
)

base_total = leaf_counts["base_quota"].sum()

print("Rows used to guarantee every leaf:", base_total)
print("Rows remaining:", TARGET_N - base_total)

In [ ]:
# ============================================================
# STEP 6: Distribute remaining sample positions
# ============================================================

remaining = TARGET_N - leaf_counts["base_quota"].sum()

if remaining < 0:
    raise ValueError(
        "TARGET_N is too small to give every leaf at least one row."
    )

# Capacity remaining after the guaranteed row
leaf_counts["remaining_capacity"] = (
    leaf_counts["leaf_population_count"]
    - leaf_counts["base_quota"]
)

total_remaining_population = (
    leaf_counts["remaining_capacity"].sum()
)

# Allocate remaining rows proportionally
leaf_counts["extra_ideal"] = (
    remaining
    * leaf_counts["remaining_capacity"]
    / total_remaining_population
)

# Floor first
leaf_counts["extra_quota"] = (
    np.floor(leaf_counts["extra_ideal"])
    .astype(int)
)

# Current quota
leaf_counts["leaf_sample_quota"] = (
    leaf_counts["base_quota"]
    + leaf_counts["extra_quota"]
)

In [ ]:
# ============================================================
# STEP 7: Make quota total EXACTLY TARGET_N
# ============================================================

current_total = leaf_counts["leaf_sample_quota"].sum()

rows_left = TARGET_N - current_total

print("Current total:", current_total)
print("Rows still needing allocation:", rows_left)

# Fractional part of the ideal allocation
leaf_counts["remainder"] = (
    leaf_counts["extra_ideal"]
    - np.floor(leaf_counts["extra_ideal"])
)

# Only leaves with remaining capacity can receive another row
eligible = (
    leaf_counts["leaf_sample_quota"]
    < leaf_counts["leaf_population_count"]
)

extra_indices = (
    leaf_counts.loc[eligible]
    .sort_values("remainder", ascending=False)
    .head(rows_left)
    .index
)

leaf_counts.loc[
    extra_indices,
    "leaf_sample_quota"
] += 1

print(
    "FINAL LEAF QUOTA TOTAL:",
    leaf_counts["leaf_sample_quota"].sum()
)

assert leaf_counts["leaf_sample_quota"].sum() == TARGET_N

In [ ]:
# ============================================================
# STEP 8: Inspect leaf quotas
# ============================================================

leaf_counts["sampling_fraction"] = (
    leaf_counts["leaf_sample_quota"]
    / leaf_counts["leaf_population_count"]
)

leaf_counts["sample_weight_leaf"] = (
    leaf_counts["leaf_population_count"]
    / leaf_counts["leaf_sample_quota"]
)

print("Smallest leaves:")
display(
    leaf_counts
    .sort_values("leaf_population_count")
    .head(30)
)

print("\nLargest leaves:")
display(
    leaf_counts
    .sort_values("leaf_population_count", ascending=False)
    .head(30)
)

In [ ]:
# ============================================================
# STEP 9: Allocate each leaf quota across its length buckets
# ============================================================

# We need joint_counts and leaf_counts from the previous cells.

# Merge leaf-level quotas into the joint table
joint_alloc = joint_counts.merge(
    leaf_counts[[
        "fp_leaf_node_id",
        "leaf_population_count",
        "leaf_sample_quota"
    ]],
    on="fp_leaf_node_id",
    how="left",
    validate="many_to_one"
)

# Ideal quota for each joint cell inside each leaf
joint_alloc["ideal_joint_quota"] = (
    joint_alloc["leaf_sample_quota"]
    * joint_alloc["population_count"]
    / joint_alloc["leaf_population_count"]
)

# Placeholder for integer quotas
joint_alloc["joint_sample_quota"] = 0

# Allocate within each leaf
for leaf_id, idx in joint_alloc.groupby("fp_leaf_node_id").groups.items():
    sub = joint_alloc.loc[idx]

    raw = sub["ideal_joint_quota"].to_numpy()
    caps = sub["population_count"].to_numpy()
    total = int(sub["leaf_sample_quota"].iloc[0])

    alloc = bounded_integer_allocation(raw, caps, total)
    joint_alloc.loc[idx, "joint_sample_quota"] = alloc

# Sanity checks
print("Total joint quota:", int(joint_alloc["joint_sample_quota"].sum()))
print("Target rows:", TARGET_N)

assert int(joint_alloc["joint_sample_quota"].sum()) == TARGET_N

# Every leaf should sum back to its leaf quota
check_leaf = (
    joint_alloc.groupby("fp_leaf_node_id", observed=True)["joint_sample_quota"]
    .sum()
    .reset_index(name="recomputed_leaf_quota")
)

leaf_check = leaf_counts[[
    "fp_leaf_node_id",
    "leaf_sample_quota"
]].merge(check_leaf, on="fp_leaf_node_id", how="left")

display(leaf_check.head(20))
assert (leaf_check["leaf_sample_quota"] == leaf_check["recomputed_leaf_quota"]).all()

display(joint_alloc.head(30))

In [ ]:
# ============================================================
# STEP 10: Sample actual rows from each joint group
# ============================================================

import pandas as pd
import numpy as np

SEED = 42

# Create quota lookup
quota_lookup = joint_alloc.set_index(
    ["fp_leaf_node_id", "length_bucket"]
)["joint_sample_quota"]

sample_parts = []

# Go through every Leaf ID x Length Bucket group
for (leaf_id, bucket), group in df.groupby(
    ["fp_leaf_node_id", "length_bucket"],
    observed=True,
    sort=False
):

    # Number of rows we need from this group
    q = int(
        quota_lookup.get(
            (leaf_id, bucket),
            0
        )
    )

    # Sample exactly q rows
    if q > 0:

        sampled_group = group.sample(
            n=q,
            random_state=SEED
        )

        sample_parts.append(sampled_group)


# Combine all sampled groups
sample_df = pd.concat(
    sample_parts,
    ignore_index=True
)


# ============================================================
# CHECK RESULTS
# ============================================================

print("Original rows:", len(df))
print("Target rows:", TARGET_N)
print("Final sample rows:", len(sample_df))

print("\nOriginal unique leaf IDs:")
print(df["fp_leaf_node_id"].nunique())

print("\nSample unique leaf IDs:")
print(sample_df["fp_leaf_node_id"].nunique())

assert len(sample_df) == TARGET_N

print("\nSUCCESS: sample contains exactly", TARGET_N, "rows")

display(sample_df.head(20))

In [ ]:
# ============================================================
# STEP 11: Add sampling weights
# ============================================================

weight_lookup = joint_alloc[
    [
        "fp_leaf_node_id",
        "length_bucket",
        "population_count",
        "joint_sample_quota"
    ]
].copy()


# Avoid division by zero
weight_lookup = weight_lookup[
    weight_lookup["joint_sample_quota"] > 0
].copy()


# Population rows represented by each sampled row
weight_lookup["sample_weight"] = (
    weight_lookup["population_count"]
    /
    weight_lookup["joint_sample_quota"]
)


# Add weights to sampled rows
sample_df = sample_df.merge(
    weight_lookup[
        [
            "fp_leaf_node_id",
            "length_bucket",
            "sample_weight"
        ]
    ],
    on=[
        "fp_leaf_node_id",
        "length_bucket"
    ],
    how="left",
    validate="many_to_one"
)


print("Sample rows after adding weights:", len(sample_df))

display(
    sample_df[
        [
            "fp_leaf_node_id",
            "ip_description",
            "description_length",
            "length_bucket",
            "sample_weight"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# STEP 12: Validate sample vs population
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

# Recompute bucket counts in original and sample
orig_bucket = (
    df["length_bucket"]
    .value_counts(sort=False)
    .sort_index()
)

samp_bucket = (
    sample_df["length_bucket"]
    .value_counts(sort=False)
    .reindex(orig_bucket.index, fill_value=0)
)

bucket_compare = pd.DataFrame({
    "original_count": orig_bucket,
    "sample_count": samp_bucket,
})

bucket_compare["original_pct"] = bucket_compare["original_count"] / len(df) * 100
bucket_compare["sample_pct"] = bucket_compare["sample_count"] / len(sample_df) * 100
bucket_compare["pct_diff"] = bucket_compare["sample_pct"] - bucket_compare["original_pct"]

print("Length bucket comparison:")
display(bucket_compare)

# Leaf comparison
orig_leaf = df["fp_leaf_node_id"].value_counts().sort_index()
samp_leaf = sample_df["fp_leaf_node_id"].value_counts().reindex(orig_leaf.index, fill_value=0)

leaf_compare = pd.DataFrame({
    "original_count": orig_leaf,
    "sample_count": samp_leaf,
})

leaf_compare["original_pct"] = leaf_compare["original_count"] / len(df) * 100
leaf_compare["sample_pct"] = leaf_compare["sample_count"] / len(sample_df) * 100
leaf_compare["pct_diff"] = leaf_compare["sample_pct"] - leaf_compare["original_pct"]

print("\nLeaf comparison:")
display(leaf_compare.sort_values("pct_diff", key=lambda s: s.abs(), ascending=False).head(30))

# Plot length bucket comparison
plt.figure(figsize=(14, 5))
plt.bar(bucket_compare.index.astype(str), bucket_compare["original_pct"], label="Original")
plt.bar(bucket_compare.index.astype(str), bucket_compare["sample_pct"], alpha=0.7, label="Sample")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Percent of rows")
plt.title("Original vs Sample: Length Buckets")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
output_csv = "updated_sample_data.csv"

# Optional: remove helper columns if they exist
helper_cols = [c for c in ["_row_id", "_priority"] if c in sample_df.columns]
sample_to_save = sample_df.drop(columns=helper_cols, errors="ignore")

sample_to_save.to_csv(output_csv, index=False)

print("Saved:", output_csv)
print("Rows:", len(sample_to_save))
print("Columns:", sample_to_save.columns.tolist())

In [ ]:
output_csv = "updated_sample_data_clean.csv"

keep_cols = [
    "fp_leaf_node_id",
    "ip_description",
    "description_length",
    "length_bucket",
    "sample_weight",
]

sample_df[keep_cols].to_csv(output_csv, index=False)

print("Saved:", output_csv)
print("Rows:", len(sample_df[keep_cols]))

In [ ]:
active_plot_df = xref_relevant_active_counts.assign(
    relationship=lambda df: df["cross_type"] + " | " + df["relationship_code_msg"].fillna("NULL")
)

ax = active_plot_df.plot.bar(
    x="relationship",
    y=["row_count", "active_ip_and_fp_rows"],
    figsize=(12, 5),
)
format_axis(ax, "Relevant XREF rows before/after active IP+FP filter", "Relationship", "Rows")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
import pandas as pd
import pycld2

# Change this to your actual CSV path
INPUT_CSV = "updated_sample_data.csv"

df = pd.read_csv(INPUT_CSV)

print("Rows loaded:", len(df))
print("Columns:")
print(df.columns.tolist())

In [ ]:
original_df = pd.read_csv("xref_relevant_examples_sample_200k.csv")
sample_df = pd.read_csv("updated_sample_data.csv")

print("Original:", len(original_df))
print("Sample:", len(sample_df))

In [ ]:
def detect_language(text):

    if pd.isna(text):
        return {
            "language": "UNKNOWN",
            "language_code": "unknown",
            "reliable": False,
            "confidence": 0.0
        }

    text = str(text).strip()

    if text == "":
        return {
            "language": "EMPTY",
            "language_code": "empty",
            "reliable": False,
            "confidence": 0.0
        }

    try:
        is_reliable, text_bytes_found, details = pycld2.detect(text)

        language_name = details[0][0]
        language_code = details[0][1]
        confidence = details[0][2]

        return {
            "language": language_name,
            "language_code": language_code,
            "reliable": is_reliable,
            "confidence": confidence
        }

    except Exception:
        return {
            "language": "ERROR",
            "language_code": "error",
            "reliable": False,
            "confidence": 0.0
        }

In [ ]:
print(
    detect_language(
        "This is a laboratory product description"
    )
)

print(
    detect_language(
        "Solution tampon pour laboratoire"
    )
)

print(
    detect_language(
        "Pufferlösung für Laboranwendungen"
    )
)

In [ ]:
language_results = df["ip_description"].apply(
    detect_language
)

language_results = pd.DataFrame(
    language_results.tolist(),
    index=df.index
)

language_results = language_results.add_prefix("lang_")

df = pd.concat(
    [
        df,
        language_results
    ],
    axis=1
)

display(
    df[
        [
            "ip_description",
            "lang_language",
            "lang_language_code",
            "lang_reliable",
            "lang_confidence"
        ]
    ].head(30)
)

In [ ]:
language_summary = (
    df.groupby(
        [
            "lang_language",
            "lang_language_code"
        ]
    )
    .agg(
        count=("ip_description", "size"),
        reliable_count=("lang_reliable", "sum")
    )
    .reset_index()
)

language_summary["percentage"] = (
    language_summary["count"]
    / len(df)
    * 100
)

language_summary["reliable_percentage"] = (
    language_summary["reliable_count"]
    / language_summary["count"]
    * 100
)

language_summary = language_summary.sort_values(
    "count",
    ascending=False
)

display(language_summary.head(30))

In [ ]:
import pandas as pd
import pycld2

def detect_language(text):
    if pd.isna(text):
        return {
            "language": "UNKNOWN",
            "language_code": "unknown",
            "reliable": False,
            "confidence": 0.0
        }

    text = str(text).strip()

    if not text:
        return {
            "language": "EMPTY",
            "language_code": "empty",
            "reliable": False,
            "confidence": 0.0
        }

    try:
        is_reliable, text_bytes_found, details = pycld2.detect(text)

        # First detected language
        language_name = details[0][0]
        language_code = details[0][1]
        confidence = details[0][2]

        return {
            "language": language_name,
            "language_code": language_code,
            "reliable": bool(is_reliable),
            "confidence": float(confidence)
        }

    except Exception:
        return {
            "language": "ERROR",
            "language_code": "error",
            "reliable": False,
            "confidence": 0.0
        }

In [ ]:
language_results = df["ip_description"].apply(detect_language)

language_results = pd.DataFrame(
    language_results.tolist(),
    index=df.index
)

df = pd.concat(
    [df, language_results.add_prefix("lang_")],
    axis=1
)

display(
    df[
        [
            "ip_description",
            "lang_language",
            "lang_language_code",
            "lang_reliable",
            "lang_confidence"
        ]
    ].head(20)
)

In [ ]:
language_counts = (
    df["lang_language"]
    .value_counts()
)

language_pct = (
    language_counts / len(df) * 100
)

language_summary = pd.DataFrame({
    "count": language_counts,
    "percentage": language_pct.round(3)
})

display(language_summary)

In [ ]:
target_languages = [
    "ENGLISH",
    "FRENCH",
    "SPANISH",
    "ITALIAN",
    "GERMAN",
]

display(
    language_summary[
        language_summary.index.isin(target_languages)
    ]
)

In [ ]:
leaf_language = pd.crosstab(
    df["fp_leaf_node_id"],
    df["lang_language"]
)

display(leaf_language.head(20))

In [ ]:
leaf_language_pct = (
    pd.crosstab(
        df["fp_leaf_node_id"],
        df["lang_language"],
        normalize="index"
    ) * 100
)

display(leaf_language_pct.head(20))

In [ ]:
sample_language_results = sample_df["ip_description"].apply(
    detect_language
)

sample_language_results = pd.DataFrame(
    sample_language_results.tolist(),
    index=sample_df.index
)

sample_df = pd.concat(
    [
        sample_df,
        sample_language_results.add_prefix("lang_")
    ],
    axis=1
)

In [ ]:
original_language = (
    df["lang_language"]
    .value_counts(normalize=True)
    * 100
)

sample_language = (
    sample_df["lang_language"]
    .value_counts(normalize=True)
    * 100
)

language_comparison = pd.DataFrame({
    "original_pct": original_language,
    "sample_pct": sample_language
}).fillna(0)

language_comparison["difference_pct_points"] = (
    language_comparison["sample_pct"]
    - language_comparison["original_pct"]
)

display(
    language_comparison.sort_values(
        "original_pct",
        ascending=False
    )
)

In [ ]:
original_leaf_lang = pd.crosstab(
    df["fp_leaf_node_id"],
    df["lang_language"],
    normalize="index"
)

sample_leaf_lang = pd.crosstab(
    sample_df["fp_leaf_node_id"],
    sample_df["lang_language"],
    normalize="index"
)

# Align rows/columns
original_leaf_lang, sample_leaf_lang = original_leaf_lang.align(
    sample_leaf_lang,
    join="outer",
    axis=0
)

original_leaf_lang, sample_leaf_lang = original_leaf_lang.align(
    sample_leaf_lang,
    join="outer",
    axis=1
)

leaf_language_difference = (
    sample_leaf_lang.fillna(0)
    - original_leaf_lang.fillna(0)
)

display(
    leaf_language_difference.head(20)
)

In [ ]:
english_labels = {
    "ENGLISH"
}

non_english_df = df[
    ~df["lang_language"].isin(english_labels)
].copy()

In [ ]:
print("Total rows:", len(df))
print("Non-English rows:", len(non_english_df))
print(
    "Non-English percentage:",
    len(non_english_df) / len(df) * 100
)

In [ ]:
display(
    non_english_df["lang_language"]
    .value_counts()
    .to_frame("count")
)

In [ ]:
non_english_languages = (
    non_english_df["lang_language"]
    .value_counts()
)

display(
    non_english_languages[
        non_english_languages.index.isin(
            ["FRENCH", "SPANISH", "ITALIAN", "GERMAN"]
        )
    ]
)

In [ ]:
evaluation_languages = [
    "FRENCH",
    "SPANISH",
    "ITALIAN",
    "GERMAN"
]

language_eval_df = df[
    df["lang_language"].isin(evaluation_languages)
].copy()

print(
    "Multilingual evaluation rows:",
    len(language_eval_df)
)

display(
    language_eval_df["lang_language"]
    .value_counts()
)

In [ ]:
# Run language detection on ALL rows
language_results = df["ip_description"].apply(detect_language)

# Convert results to DataFrame
language_results = pd.DataFrame(
    language_results.tolist(),
    index=df.index
)

language_results.columns = [
    "language",
    "language_code",
    "language_reliable",
    "language_confidence"
]

# Add language columns to original dataset
df = pd.concat(
    [df, language_results],
    axis=1
)

# Save ALL rows to CSV
OUTPUT_CSV = "200k_dataset_with_language.csv"

df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("Language detection complete.")
print("CSV saved:", OUTPUT_CSV)
print("Total rows saved:", len(df))
print("Total columns saved:", len(df.columns))

In [ ]:
language_results = df["ip_description"].apply(detect_language)

In [ ]:
%pip install lingua-language-detector

In [ ]:
import pandas as pd

from lingua import (
    Language,
    LanguageDetectorBuilder
)

In [ ]:
TARGET_LANGUAGES = [
    Language.ENGLISH,
    Language.FRENCH,
    Language.ITALIAN,
    Language.SPANISH,
    Language.GERMAN
]

detector = (
    LanguageDetectorBuilder
    .from_languages(*TARGET_LANGUAGES)
    .build()
)

In [ ]:
MIN_CONFIDENCE = 0.60
MIN_MARGIN = 0.15

def detect_target_language(text):
    
    if pd.isna(text):
        return {
            "restricted_language": "UNKNOWN",
            "restricted_code": "unknown",
            "restricted_confidence": 0.0,
            "restricted_margin": 0.0,
            "restricted_status": "MISSING"
        }

    text = str(text).strip()

    if text == "":
        return {
            "restricted_language": "UNKNOWN",
            "restricted_code": "unknown",
            "restricted_confidence": 0.0,
            "restricted_margin": 0.0,
            "restricted_status": "EMPTY"
        }

    try:
        confidence_values = (
            detector.compute_language_confidence_values(text)
        )

        top1 = confidence_values[0]
        top2 = confidence_values[1]

        top_language = top1.language
        top_confidence = float(top1.value)

        second_confidence = float(top2.value)

        margin = (
            top_confidence
            - second_confidence
        )

        language_name = top_language.name

        language_code = (
            top_language
            .iso_code_639_1
            .name
            .lower()
        )

        # Don't force ambiguous descriptions
        if (
            top_confidence < MIN_CONFIDENCE
            or margin < MIN_MARGIN
        ):
            final_language = "UNCERTAIN"
            status = "LOW_CONFIDENCE"

        else:
            final_language = language_name
            status = "ACCEPTED"

        return {
            "restricted_language": final_language,
            "restricted_code": language_code,
            "restricted_confidence": top_confidence,
            "restricted_margin": margin,
            "restricted_status": status
        }

    except Exception:
        return {
            "restricted_language": "ERROR",
            "restricted_code": "error",
            "restricted_confidence": 0.0,
            "restricted_margin": 0.0,
            "restricted_status": "ERROR"
        }

In [ ]:
restricted_results = (
    df["ip_description"]
    .apply(detect_target_language)
)

restricted_results = pd.DataFrame(
    restricted_results.tolist(),
    index=df.index
)

df = pd.concat(
    [
        df,
        restricted_results
    ],
    axis=1
)

print("Restricted language detection complete.")
print("Rows processed:", len(df))

In [ ]:
TARGET_CODES = {
    "en",
    "fr",
    "it",
    "es",
    "de"
}

In [ ]:
def create_final_language(row):

    cld_code = str(
        row["language_code"]
    ).lower()

    cld_reliable = bool(
        row["language_reliable"]
    )

    restricted = row[
        "restricted_language"
    ]

    # pycld2 confidently believes it is
    # a language outside our five
    if (
        cld_reliable
        and cld_code not in TARGET_CODES
        and cld_code not in {
            "unknown",
            "error",
            "un"
        }
    ):
        return "OTHER"

    # Restricted detector was uncertain
    if restricted in {
        "UNCERTAIN",
        "UNKNOWN",
        "ERROR"
    }:
        return "UNCERTAIN"

    return restricted

In [ ]:
df["final_language"] = df.apply(
    create_final_language,
    axis=1
)

In [ ]:
final_language_counts = (
    df["final_language"]
    .value_counts()
)

final_language_summary = pd.DataFrame({
    "count": final_language_counts,
    "percentage": (
        final_language_counts
        / len(df)
        * 100
    ).round(3)
})

display(final_language_summary)

In [ ]:
confidence_summary = (
    df.groupby("final_language")
    .agg(
        count=("ip_description", "size"),
        avg_confidence=(
            "restricted_confidence",
            "mean"
        ),
        avg_margin=(
            "restricted_margin",
            "mean"
        )
    )
    .sort_values(
        "count",
        ascending=False
    )
)

display(confidence_summary)

In [ ]:
comparison = pd.crosstab(
    df["language"],
    df["final_language"]
)

display(comparison)

In [ ]:
target_code_to_name = {
    "en": "ENGLISH",
    "fr": "FRENCH",
    "it": "ITALIAN",
    "es": "SPANISH",
    "de": "GERMAN"
}

df["pycld2_target_language"] = (
    df["language_code"]
    .str.lower()
    .map(target_code_to_name)
)

disagreements = df[
    df["pycld2_target_language"].notna()
    &
    (
        df["pycld2_target_language"]
        != df["restricted_language"]
    )
].copy()

print("Target-language disagreements:", len(disagreements))

display(
    disagreements[
        [
            "ip_description",
            "language",
            "language_reliable",
            "language_confidence",
            "restricted_language",
            "restricted_confidence",
            "restricted_margin"
        ]
    ].head(50)
)

In [ ]:
OUTPUT_CSV = "200k_language_analysis_restricted.csv"

df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("Saved:", OUTPUT_CSV)
print("Rows saved:", len(df))

### Questions

- For active `COMP2FISH`, how many rows have non-null `IP_DESCRIPTION`, and what is the description-length distribution?
- For active `COMP2FISH` and `FISH2FISH`, how many inputs have more than one output match?
- For active `FISH2FISH`, how many rows have equal vs non-equal `FP_NUMBER` and `IP_NUMBER`?

In [ ]:
comp2fish_ip_description_length_distribution = query_df("""
    WITH active_comp2fish AS (
        SELECT ip_description
        FROM XREF_ADMIN.XREF_COMP_VEND_FISH_EU_VW
        WHERE cross_type = 'COMP2FISH'
          AND (ip_product_type IS NULL OR ip_product_type NOT IN ('04', '07'))
          AND (ip_discontinued_code IS NULL OR ip_discontinued_code NOT IN ('M', 'O'))
          AND ip_restriction_code = 'RTSN'
          AND (fp_product_type IS NULL OR fp_product_type NOT IN ('04', '07'))
          AND (fp_discontinued_code IS NULL OR fp_discontinued_code NOT IN ('M', 'O'))
          AND fp_restriction_code = 'RTSN'
    )
    SELECT
        CASE
            WHEN ip_description IS NULL THEN 'NULL'
            WHEN LENGTH(ip_description) >= 200 THEN '200+'
            ELSE TO_CHAR(FLOOR(LENGTH(ip_description) / 10) * 10) || '-' || TO_CHAR(FLOOR(LENGTH(ip_description) / 10) * 10 + 9)
        END AS description_length_bucket,
        CASE
            WHEN ip_description IS NULL THEN -1
            WHEN LENGTH(ip_description) >= 200 THEN 200
            ELSE FLOOR(LENGTH(ip_description) / 10) * 10
        END AS bucket_sort,
        COUNT(*) AS row_count,
        SUM(CASE WHEN ip_description IS NOT NULL THEN 1 ELSE 0 END) AS non_null_description_rows
    FROM active_comp2fish
    GROUP BY
        CASE
            WHEN ip_description IS NULL THEN 'NULL'
            WHEN LENGTH(ip_description) >= 200 THEN '200+'
            ELSE TO_CHAR(FLOOR(LENGTH(ip_description) / 10) * 10) || '-' || TO_CHAR(FLOOR(LENGTH(ip_description) / 10) * 10 + 9)
        END,
        CASE
            WHEN ip_description IS NULL THEN -1
            WHEN LENGTH(ip_description) >= 200 THEN 200
            ELSE FLOOR(LENGTH(ip_description) / 10) * 10
        END
    ORDER BY bucket_sort
""")

save_df(comp2fish_ip_description_length_distribution, "xref_comp_vend_fish_comp2fish_ip_description_length_distribution.csv")
display(comp2fish_ip_description_length_distribution)

In [ ]:
ax = comp2fish_ip_description_length_distribution.plot.bar(
    x="description_length_bucket",
    y="row_count",
    figsize=(14, 5),
    legend=False,
    color="tab:blue",
)
format_axis(ax, "Active COMP2FISH IP description length distribution", "Description length bucket", "Rows")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
xref_input_output_match_distribution = query_df("""
    WITH active_relevant AS (
        SELECT
            cross_type,
            ip_number,
            fp_number
        FROM XREF_ADMIN.XREF_COMP_VEND_FISH_EU_VW
        WHERE cross_type IN ('COMP2FISH', 'FISH2FISH')
          AND (ip_product_type IS NULL OR ip_product_type NOT IN ('04', '07'))
          AND (ip_discontinued_code IS NULL OR ip_discontinued_code NOT IN ('M', 'O'))
          AND ip_restriction_code = 'RTSN'
          AND (fp_product_type IS NULL OR fp_product_type NOT IN ('04', '07'))
          AND (fp_discontinued_code IS NULL OR fp_discontinued_code NOT IN ('M', 'O'))
          AND fp_restriction_code = 'RTSN'
    ), per_input AS (
        SELECT
            cross_type,
            ip_number,
            COUNT(DISTINCT fp_number) AS distinct_fp_matches
        FROM active_relevant
        GROUP BY cross_type, ip_number
    )
    SELECT
        cross_type,
        CASE WHEN distinct_fp_matches >= 5 THEN '5+' ELSE TO_CHAR(distinct_fp_matches) END AS fp_match_count_bucket,
        CASE WHEN distinct_fp_matches >= 5 THEN 5 ELSE distinct_fp_matches END AS bucket_sort,
        COUNT(*) AS distinct_ip_numbers
    FROM per_input
    GROUP BY
        cross_type,
        CASE WHEN distinct_fp_matches >= 5 THEN '5+' ELSE TO_CHAR(distinct_fp_matches) END,
        CASE WHEN distinct_fp_matches >= 5 THEN 5 ELSE distinct_fp_matches END
    ORDER BY cross_type, bucket_sort
""")

save_df(xref_input_output_match_distribution, "xref_comp_vend_fish_input_output_match_distribution.csv")
display(xref_input_output_match_distribution)

In [ ]:
pivot_df = xref_input_output_match_distribution.pivot(
    index="fp_match_count_bucket",
    columns="cross_type",
    values="distinct_ip_numbers",
).fillna(0)

ax = pivot_df.plot.bar(figsize=(10, 5))
format_axis(ax, "Active inputs by number of distinct FP matches", "Distinct FP matches per IP", "Distinct IP numbers")
plt.show()

In [ ]:
fish2fish_ip_fp_equality = query_df("""
    SELECT
        CASE WHEN ip_number = fp_number THEN 'IP_EQUALS_FP' ELSE 'IP_DIFFERS_FROM_FP' END AS equality_status,
        COUNT(*) AS row_count,
        COUNT(DISTINCT ip_number) AS distinct_ip_numbers,
        COUNT(DISTINCT fp_number) AS distinct_fp_numbers
    FROM XREF_ADMIN.XREF_COMP_VEND_FISH_EU_VW
    WHERE cross_type = 'FISH2FISH'
      AND (ip_product_type IS NULL OR ip_product_type NOT IN ('04', '07'))
      AND (ip_discontinued_code IS NULL OR ip_discontinued_code NOT IN ('M', 'O'))
      AND ip_restriction_code = 'RTSN'
      AND (fp_product_type IS NULL OR fp_product_type NOT IN ('04', '07'))
      AND (fp_discontinued_code IS NULL OR fp_discontinued_code NOT IN ('M', 'O'))
      AND fp_restriction_code = 'RTSN'
    GROUP BY CASE WHEN ip_number = fp_number THEN 'IP_EQUALS_FP' ELSE 'IP_DIFFERS_FROM_FP' END
    ORDER BY row_count DESC
""")

save_df(fish2fish_ip_fp_equality, "xref_comp_vend_fish_fish2fish_ip_fp_equality.csv")
display(fish2fish_ip_fp_equality)

In [ ]:
ax = fish2fish_ip_fp_equality.plot.bar(
    x="equality_status",
    y="row_count",
    figsize=(7, 5),
    legend=False,
    color="tab:orange",
)
format_axis(ax, "Active FISH2FISH IP/FP equality", "Status", "Rows")
plt.xticks(rotation=0)
plt.show()

Image ocr

In [ ]:
%pip install easyocr opencv-python-headless pillow openai pydantic

In [ ]:
import os
import re
import json
import base64
import io

import numpy as np
import easyocr

from PIL import Image
from openai import OpenAI

In [ ]:
IMAGE_PATH = "product-label-image.webp"

image = Image.open(IMAGE_PATH).convert("RGB")

display(image)

print("Image size:", image.size)

In [ ]:
reader = easyocr.Reader(
    ["en"],
    gpu=False
)

print("EasyOCR loaded.")

In [ ]:
def run_ocr(image):
    """
    Extract text from a PIL image using EasyOCR.

    Returns:
        ocr_text: combined OCR text
        ocr_details: OCR text + confidence + bounding boxes
    """

    image_array = np.array(image)

    results = reader.readtext(
        image_array,
        detail=1,
        paragraph=False
    )

    ocr_details = []
    text_lines = []

    for bbox, text, confidence in results:

        text = text.strip()

        if not text:
            continue

        text_lines.append(text)

        ocr_details.append({
            "text": text,
            "confidence": float(confidence),
            "bbox": bbox
        })

    ocr_text = "\n".join(text_lines)

    return ocr_text, ocr_details

In [ ]:
ocr_text, ocr_details = run_ocr(image)

print("========== OCR TEXT ==========")
print(ocr_text)
print("==============================")

In [ ]:
for item in ocr_details:
    print(
        f"{item['confidence']:.3f}  |  {item['text']}"
    )

In [ ]:
agent_result = extract_product_information(
    ocr_text
)

print(
    json.dumps(
        agent_result,
        indent=4
    )
)

In [ ]:
def process_image(image):

    final_response = {
        "error_msg": "",
        "success": False,
        "ocr_text": None,
        "ocr_pn": "",
        "ocr_description": ""
    }

    # -------------------------
    # STEP 1: OCR
    # -------------------------

    try:
        ocr_text, ocr_details = run_ocr(
            image
        )

    except Exception as e:

        final_response["error_msg"] = (
            f"OCR failed: {str(e)}"
        )

        return final_response

    final_response["ocr_text"] = ocr_text

    if not ocr_text.strip():

        final_response["error_msg"] = (
            "OCR completed but no text was detected."
        )

        return final_response

    # -------------------------
    # STEP 2: Agent
    # -------------------------

    extraction = extract_product_information(
        ocr_text
    )

    final_response["error_msg"] = (
        extraction["error_msg"]
    )

    final_response["success"] = (
        extraction["success"]
    )

    final_response["ocr_pn"] = (
        extraction["ocr_pn"]
    )

    final_response["ocr_description"] = (
        extraction["ocr_description"]
    )

    return final_response

In [ ]:
result = process_image(image)

print(
    json.dumps(
        result,
        indent=4,
        ensure_ascii=False
    )
)

In [ ]:
%pip install openai python-dotenv

In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI

# Load variables from .env
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY was not found. Check your .env file."
    )

client = OpenAI(
    api_key=api_key,
    timeout=30.0,
    max_retries=0
)

MODEL = "gpt-5.6-luna"

print("OpenAI client configured successfully.")
print("Model:", MODEL)

In [ ]:
try:
    response = client.responses.create(
        model=MODEL,
        input="Reply with exactly: API connection successful"
    )

    print(response.output_text)

except Exception as e:
    print("API ERROR:")
    print(type(e).__name__)
    print(str(e))

In [ ]:
import time

print("Checking image...")

print("Image type:", type(image))
print("Image size:", image.size)

display(image)

print("\nStarting OCR...")

start = time.time()

try:
    ocr_text, ocr_details = run_ocr(image)

    print(
        f"OCR finished in {time.time() - start:.2f} seconds"
    )

    print("\n========== OCR TEXT ==========")
    print(ocr_text)
    print("==============================")

except Exception as e:
    print("\nOCR ERROR")
    print(type(e).__name__)
    print(str(e))

In [ ]:
import time

print("Testing OpenAI API...")

start = time.time()

try:
    response = client.responses.create(
        model="gpt-5.6-luna",
        input="Reply with exactly: API working",
        max_output_tokens=20
    )

    print(
        f"API finished in {time.time() - start:.2f} seconds"
    )

    print(response.output_text)

except Exception as e:
    print("\nOPENAI API ERROR")
    print(type(e).__name__)
    print(str(e))

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=30.0,
    max_retries=0
)

MODEL = "gpt-5.6-luna"

In [ ]:
import json
import re

def extract_product_information(ocr_text):

    if not ocr_text or not ocr_text.strip():
        return {
            "error_msg": "OCR produced no text.",
            "success": False,
            "ocr_pn": "",
            "ocr_description": ""
        }

    prompt = f"""
You are a product label information extraction agent.

You receive OCR text extracted from a product image.

Extract:
1. Product Number / Part Number
2. Product Description

OCR TEXT:
----------------
{ocr_text}
----------------

Rules:
- Only use information present in the OCR text.
- Do not invent values.
- Preserve the part number as closely as possible.
- Do not confuse quantity, serial number, lot number, date,
  barcode, or country information with the part number.
- If multiple possible part numbers exist and it is unclear which
  one is correct, set success=false and explain why in error_msg.
- If the PN is missing, leave ocr_pn empty.
- If the description is missing, leave ocr_description empty.
- error_msg should be empty when the extraction is clear.

Return ONLY valid JSON in exactly this format:

{{
    "error_msg": "",
    "success": true,
    "ocr_pn": "",
    "ocr_description": ""
}}
"""

    try:
        response = client.responses.create(
            model=MODEL,
            input=prompt
        )

        raw_result = response.output_text.strip()

        # Remove markdown code fences if returned
        raw_result = re.sub(
            r"^```(?:json)?\\s*",
            "",
            raw_result,
            flags=re.IGNORECASE
        )

        raw_result = re.sub(
            r"\\s*```$",
            "",
            raw_result
        )

        data = json.loads(raw_result)

        return {
            "error_msg": str(data.get("error_msg", "")),
            "success": bool(data.get("success", False)),
            "ocr_pn": str(data.get("ocr_pn", "") or ""),
            "ocr_description": str(
                data.get("ocr_description", "") or ""
            )
        }

    except Exception as e:
        return {
            "error_msg": f"Agent extraction failed: {str(e)}",
            "success": False,
            "ocr_pn": "",
            "ocr_description": ""
        }

In [ ]:
import json

ocr_text, ocr_details = run_ocr(image)

result = extract_product_information(
    ocr_text
)

final_response = {
    "error_msg": result.get("error_msg", ""),
    "success": result.get("success", False),
    "ocr_text": ocr_text,
    "ocr_pn": result.get("ocr_pn", ""),
    "ocr_description": result.get(
        "ocr_description",
        ""
    )
}

print(
    json.dumps(
        final_response,
        indent=4,
        ensure_ascii=False
    )
)

## IMAGE_OCR_TEST

In [ ]:
%pip install --upgrade pip

In [ ]:
%pip install easyocr opencv-python-headless pillow openai python-dotenv

### Imports

In [ ]:
import os
import re
import json
import time

import numpy as np
import easyocr

from PIL import Image
from openai import OpenAI
from dotenv import load_dotenv

### Initialize EasyOCR

In [ ]:
print("Loading EasyOCR...")

reader = easyocr.Reader(
    ["en"],
    gpu=False
)

print("EasyOCR ready.")

### Define run_ocr()

In [ ]:
def run_ocr(image):
    """
    Run EasyOCR on a PIL image.

    Returns:
        ocr_text:
            All detected text combined with newlines.

        ocr_details:
            Individual OCR results including confidence
            and bounding boxes.
    """

    image_array = np.array(image)

    results = reader.readtext(
        image_array,
        detail=1,
        paragraph=False
    )

    text_lines = []
    ocr_details = []

    for bbox, text, confidence in results:

        text = str(text).strip()

        if not text:
            continue

        text_lines.append(text)

        ocr_details.append({
            "text": text,
            "confidence": float(confidence),
            "bbox": bbox
        })

    ocr_text = "\n".join(text_lines)

    return ocr_text, ocr_details

### Load image

In [ ]:
IMAGE_PATH = "IMAGES/Media 1.jpg"

image = Image.open(
    IMAGE_PATH
).convert("RGB")

print("Image loaded successfully.")
print("Image size:", image.size)

display(image)

### Run OCR → ocr_text

In [ ]:
print("Starting OCR...")

start = time.time()

ocr_text, ocr_details = run_ocr(
    image
)

ocr_time = time.time() - start

print(
    f"OCR finished in {ocr_time:.2f} seconds"
)

print("\n========== OCR TEXT ==========")
print(ocr_text)
print("==============================")

### Initialize OpenAI client

In [ ]:
load_dotenv()

api_key = os.getenv(
    "OPENAI_API_KEY"
)

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY was not found. "
        "Check your .env file."
    )

client = OpenAI(
    api_key=api_key,
    timeout=60.0,
    max_retries=1
)

MODEL = "gpt-5.6-luna"

print("OpenAI client configured.")
print("Model:", MODEL)

### CONNECTION TEST

In [ ]:
print("Testing API connection...")

try:

    test_response = client.responses.create(
        model=MODEL,
        input="Reply with exactly: API working",
        max_output_tokens=30
    )

    print(test_response.output_text)

except Exception as e:

    print("API ERROR:")
    print(type(e).__name__)
    print(str(e))

### Run extraction

In [ ]:
print(
    "Sending OCR text to extraction agent..."
)

start = time.time()

result = extract_product_information(
    ocr_text
)

agent_time = time.time() - start

print(
    f"Agent finished in "
    f"{agent_time:.2f} seconds"
)

print("\n========== AGENT RESULT ==========")

print(
    json.dumps(
        result,
        indent=4,
        ensure_ascii=False
    )
)

### Build final JSON response

In [ ]:
final_response = {
    "error_msg": result.get(
        "error_msg",
        ""
    ),

    "success": result.get(
        "success",
        False
    ),

    "ocr_text": ocr_text,

    "ocr_pn": result.get(
        "ocr_pn",
        []
    ),

    "ocr_description": result.get(
        "ocr_description",
        ""
    )
}

print(
    json.dumps(
        final_response,
        indent=4,
        ensure_ascii=False
    )
)

### Define extract_product_information()

In [ ]:
def extract_product_information(ocr_text):

    if not ocr_text or not ocr_text.strip():

        return {
            "error_msg": "OCR produced no text.",
            "success": False,
            "ocr_pn": [],
            "ocr_description": "",
            "reasoning": ""
        }

    instructions = """
You are a product-label information extraction agent.

You receive raw text that was extracted from an image using OCR.

Your task is to extract:

1. ALL plausible product number / part number candidates.
2. The most useful product description that can be constructed
   from information explicitly present in the OCR text.
3. A short extraction rationale for debugging and prompt tuning.


==================================================
PART NUMBER RULES
==================================================

ocr_pn MUST always be an array of strings.

There may be:
- zero candidates
- one candidate
- multiple candidates


EXPLICIT PART-NUMBER LABELS ARE VERY STRONG EVIDENCE.

Labels such as:

- Cat No.
- Cat. No.
- Catalog No.
- Catalog Number
- Catalogue Number
- Part No.
- Part Number
- P/N
- PN
- Product No.
- Product Number
- Item No.
- Item Number
- SKU
- Material No.
- Material Number

indicate that the value directly following or clearly associated
with that label is a part/product number.

When such an explicit label is present:

- include its associated value in ocr_pn
- place explicitly labeled PN candidates FIRST
- do not discard it simply because other numbers are present


OTHER POSSIBLE PN CANDIDATES

A label may contain other numbers that are not explicitly labeled
but could reasonably be product identifiers.

Include them when there is meaningful evidence they could be
product/part/catalog numbers.

A downstream database lookup will check every candidate,
so when genuinely uncertain between multiple plausible product
identifiers, returning multiple reasonable candidates is preferred
to arbitrarily choosing one.


PRESERVATION RULES

- Preserve identifiers as closely as possible to the OCR text.
- Do not invent digits or letters.
- Do not automatically add hyphens.
- Do not automatically remove meaningful spaces.
- Remove duplicate candidates.
- Preserve candidate order where possible.


NORMALLY DO NOT INCLUDE:

- quantities
- prices
- dates
- lot numbers
- batch numbers
- serial numbers
- phone numbers
- dimensions
- postal codes
- regulatory certification numbers
- country codes
- random barcode numbers when there is no evidence they represent
  a product identifier

If one of these values genuinely appears to function as the
product identifier, it may still be included.


==================================================
DESCRIPTION RULES
==================================================

Use ALL useful information in the OCR text when creating
ocr_description.

Do NOT simply select one line if other OCR text contains useful
product-identifying information.

The description may combine useful information such as:

- manufacturer
- brand
- product name
- product family
- model/type
- material
- size
- capacity
- concentration
- formulation
- specification
- variant
- color
- important product-specific attributes

ONLY include information that is actually present in the OCR text.

The goal is to produce a useful searchable product description,
not merely copy the shortest product-name line.


EXAMPLE

OCR:

ACME Scientific
Cat No. ABC-1234
UltraPure Buffer Solution
500 mL
pH 7.0
Made in Germany
CE

Good result:

ocr_pn:
["ABC-1234"]

ocr_description:
"ACME Scientific UltraPure Buffer Solution, 500 mL, pH 7.0"

The volume and pH are useful product attributes.

"Made in Germany" and "CE" are not useful description attributes
and should normally be excluded.


ANOTHER EXAMPLE

OCR:

HYUNDAI
Genuine Parts
26101 23933
COOLANT PUMP
1 PC
MADE IN KOREA
EAC

A reasonable description is:

"HYUNDAI Genuine Parts COOLANT PUMP"

Do not automatically include:
"1 PC"
"MADE IN KOREA"
"EAC"

because those values describe packaging/origin/regulatory
information rather than what the product is.


DESCRIPTION EXCLUSIONS

Normally exclude:

- barcode text
- certification marks
- regulatory marks
- country-of-origin statements
- shipping information
- addresses
- phone numbers
- website URLs
- dates
- lot/batch information

unless unusually important for identifying the actual product.

Do not invent information.


==================================================
REASONING FIELD
==================================================

Return a short field called "reasoning".

This is a concise extraction rationale intended for debugging
and prompt tuning.

It is NOT a long step-by-step analysis.

Good reasoning example:

"ABC-1234 immediately follows the explicit 'Cat No.' label, so it
was ranked first as the PN candidate. The description combines the
brand, product name, volume, and pH because those fields describe
the product; country-of-origin and CE text were excluded."

Keep reasoning concise and factual.


==================================================
ERROR HANDLING
==================================================

Multiple PN candidates are NOT an error.

error_msg should normally be an empty string.

Use error_msg when there is a real problem such as:

- OCR text is heavily corrupted
- conflicting explicit PN labels make the result unclear
- no useful product information can be identified
- essential text is incomplete or ambiguous

success should be true when useful PN candidate(s) and/or
a useful description were extracted.

success should be false when no useful product information
can confidently be extracted.
"""

    user_input = f"""
RAW OCR TEXT
============
{ocr_text}
============
"""

    schema = {
        "type": "object",
        "properties": {
            "error_msg": {
                "type": "string"
            },
            "success": {
                "type": "boolean"
            },
            "ocr_pn": {
                "type": "array",
                "items": {
                    "type": "string"
                }
            },
            "ocr_description": {
                "type": "string"
            },
            "reasoning": {
                "type": "string"
            }
        },
        "required": [
            "error_msg",
            "success",
            "ocr_pn",
            "ocr_description",
            "reasoning"
        ],
        "additionalProperties": False
    }

    try:

        response = client.responses.create(
            model=MODEL,
            instructions=instructions,
            input=user_input,
            max_output_tokens=600,
            store=False,
            text={
                "format": {
                    "type": "json_schema",
                    "name": "product_label_extraction",
                    "schema": schema,
                    "strict": True
                }
            }
        )

        data = json.loads(
            response.output_text
        )

        # ---------------------------------
        # Ensure ocr_pn is always a list
        # ---------------------------------

        pn_candidates = data.get(
            "ocr_pn",
            []
        )

        if isinstance(pn_candidates, str):
            pn_candidates = [
                pn_candidates
            ]

        if not isinstance(pn_candidates, list):
            pn_candidates = []

        # ---------------------------------
        # Clean and deduplicate PNs
        # ---------------------------------

        cleaned_pns = []

        for pn in pn_candidates:

            pn = str(pn).strip()

            if (
                pn
                and pn not in cleaned_pns
            ):
                cleaned_pns.append(pn)

        description = str(
            data.get(
                "ocr_description",
                ""
            ) or ""
        ).strip()

        error_msg = str(
            data.get(
                "error_msg",
                ""
            ) or ""
        ).strip()

        reasoning = str(
            data.get(
                "reasoning",
                ""
            ) or ""
        ).strip()

        success = bool(
            data.get(
                "success",
                False
            )
        )

        # Safety check:
        # no PN + no description means extraction
        # shouldn't be considered successful.
        if (
            not cleaned_pns
            and not description
        ):
            success = False

        return {
            "error_msg": error_msg,
            "success": success,
            "ocr_pn": cleaned_pns,
            "ocr_description": description,
            "reasoning": reasoning
        }

    except Exception as e:

        return {
            "error_msg": (
                "Agent extraction failed: "
                + str(e)
            ),
            "success": False,
            "ocr_pn": [],
            "ocr_description": "",
            "reasoning": ""
        }

### RUN_EXTRACTION

In [ ]:
print(
    "Sending OCR text to extraction agent..."
)

start = time.time()

result = extract_product_information(
    ocr_text
)

agent_seconds = time.time() - start

print(
    f"Agent finished in "
    f"{agent_seconds:.2f} seconds"
)

print("\n========== EXTRACTION ==========")

print(
    json.dumps(
        result,
        indent=4,
        ensure_ascii=False
    )
)

### Build final JSON response

In [ ]:
final_response = {
    "error_msg": result.get(
        "error_msg",
        ""
    ),

    "success": result.get(
        "success",
        False
    ),

    "ocr_text": ocr_text,

    "ocr_pn": result.get(
        "ocr_pn",
        []
    ),

    "ocr_description": result.get(
        "ocr_description",
        ""
    ),

    "reasoning": result.get(
        "reasoning",
        ""
    )
}

print(
    json.dumps(
        final_response,
        indent=4,
        ensure_ascii=False
    )
)

### Convert an image to Base64

In [ ]:
def image_file_to_base64(image_path):

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    return base64.b64encode(
        image_bytes
    ).decode("utf-8")

In [ ]:
def base64_to_image(image_base64):

    if not image_base64:
        raise ValueError(
            "No image was provided."
        )

    # Handle data:image/png;base64,...
    if "," in image_base64:
        image_base64 = image_base64.split(
            ",",
            1
        )[1]

    image_bytes = base64.b64decode(
        image_base64
    )

    return Image.open(
        io.BytesIO(image_bytes)
    ).convert("RGB")

### Complete API-style processing function

In [ ]:
def process_ocr_request(request):

    final_response = {
        "error_msg": "",
        "success": False,
        "ocr_text": None,
        "ocr_pn": [],
        "ocr_description": "",
        "reasoning": ""
    }

    # --------------------------------
    # Validate request
    # --------------------------------

    if not isinstance(request, dict):

        final_response["error_msg"] = (
            "Request must be a dictionary."
        )

        return final_response

    image_base64 = request.get(
        "image"
    )

    if not image_base64:

        final_response["error_msg"] = (
            "No image was provided."
        )

        return final_response

    # --------------------------------
    # Decode image
    # --------------------------------

    try:

        image = base64_to_image(
            image_base64
        )

    except Exception as e:

        final_response["error_msg"] = (
            "Image decoding failed: "
            + str(e)
        )

        return final_response

    # --------------------------------
    # OCR
    # --------------------------------

    try:

        ocr_text, ocr_details = run_ocr(
            image
        )

        final_response["ocr_text"] = (
            ocr_text
        )

    except Exception as e:

        final_response["error_msg"] = (
            "OCR failed: "
            + str(e)
        )

        return final_response

    if not ocr_text.strip():

        final_response["error_msg"] = (
            "OCR completed but no text "
            "was detected."
        )

        return final_response

    # --------------------------------
    # AI extraction
    # --------------------------------

    extraction = (
        extract_product_information(
            ocr_text
        )
    )

    final_response["error_msg"] = (
        extraction["error_msg"]
    )

    final_response["success"] = (
        extraction["success"]
    )

    final_response["ocr_pn"] = (
        extraction["ocr_pn"]
    )

    final_response["ocr_description"] = (
        extraction["ocr_description"]
    )

    final_response["reasoning"] = (
        extraction["reasoning"]
    )

    return final_response

### Test the complete Base64 request → response flow

In [ ]:
import base64
import io

In [ ]:
def image_file_to_base64(image_path):
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    return base64.b64encode(
        image_bytes
    ).decode("utf-8")


def base64_to_image(image_base64):
    if not image_base64:
        raise ValueError("No image was provided.")

    if "," in image_base64:
        image_base64 = image_base64.split(",", 1)[1]

    image_bytes = base64.b64decode(image_base64)

    return Image.open(
        io.BytesIO(image_bytes)
    ).convert("RGB")

In [ ]:
request = {
    "image": image_file_to_base64(
        IMAGE_PATH
    )
}

response = process_ocr_request(
    request
)

print(
    json.dumps(
        response,
        indent=4,
        ensure_ascii=False
    )
)

## XREF_COMP_PART_MASTER_EU_VW

Potential source of additional description data for external/input parts used in `COMP2FISH` rows.

In [ ]:
comp_part_master_stats = query_df("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT part_number) AS distinct_part_numbers,
        SUM(CASE WHEN part_description IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_non_null_descriptions,
        COUNT(DISTINCT CASE WHEN part_description IS NOT NULL THEN part_number END) AS distinct_parts_with_non_null_descriptions,
        AVG(LENGTH(part_description)) AS avg_description_length,
        SUM(CASE WHEN REGEXP_LIKE(part_description, '[^ -~]') THEN 1 ELSE 0 END) AS non_ascii_description_rows
    FROM XREF_ADMIN.XREF_COMP_PART_MASTER_EU_VW
""")

comp_part_master_stats = add_rate(comp_part_master_stats, "rows_with_non_null_descriptions", "total_rows", "description_row_rate")
save_df(comp_part_master_stats, "xref_comp_part_master_stats.csv")
display(comp_part_master_stats)

In [ ]:
comp_part_master_description_length_distribution = query_df("""
    SELECT
        CASE
            WHEN part_description IS NULL THEN 'NULL'
            WHEN LENGTH(part_description) >= 200 THEN '200+'
            ELSE TO_CHAR(FLOOR(LENGTH(part_description) / 10) * 10) || '-' || TO_CHAR(FLOOR(LENGTH(part_description) / 10) * 10 + 9)
        END AS description_length_bucket,
        CASE
            WHEN part_description IS NULL THEN -1
            WHEN LENGTH(part_description) >= 200 THEN 200
            ELSE FLOOR(LENGTH(part_description) / 10) * 10
        END AS bucket_sort,
        COUNT(*) AS row_count,
        COUNT(DISTINCT part_number) AS distinct_part_numbers
    FROM XREF_ADMIN.XREF_COMP_PART_MASTER_EU_VW
    GROUP BY
        CASE
            WHEN part_description IS NULL THEN 'NULL'
            WHEN LENGTH(part_description) >= 200 THEN '200+'
            ELSE TO_CHAR(FLOOR(LENGTH(part_description) / 10) * 10) || '-' || TO_CHAR(FLOOR(LENGTH(part_description) / 10) * 10 + 9)
        END,
        CASE
            WHEN part_description IS NULL THEN -1
            WHEN LENGTH(part_description) >= 200 THEN 200
            ELSE FLOOR(LENGTH(part_description) / 10) * 10
        END
    ORDER BY bucket_sort
""")

save_df(comp_part_master_description_length_distribution, "xref_comp_part_master_description_length_distribution.csv")
display(comp_part_master_description_length_distribution)

In [ ]:
ax = comp_part_master_description_length_distribution.plot.bar(
    x="description_length_bucket",
    y="row_count",
    figsize=(14, 5),
    legend=False,
    color="tab:green",
)
format_axis(ax, "COMP part master description length distribution", "Description length bucket", "Rows")
plt.xticks(rotation=45, ha="right")
plt.show()

### Questions

- How many active `COMP2FISH` IPs have null `IP_DESCRIPTION`?
- Of those, how many can be recovered from `XREF_COMP_PART_MASTER_EU_VW`?

In [ ]:
comp2fish_null_ip_description_recovery = query_df("""
    WITH active_comp2fish_null_ip_desc AS (
        SELECT DISTINCT ip_number
        FROM XREF_ADMIN.XREF_COMP_VEND_FISH_EU_VW
        WHERE cross_type = 'COMP2FISH'
          AND ip_description IS NULL
          AND ip_number IS NOT NULL
          AND (ip_product_type IS NULL OR ip_product_type NOT IN ('04', '07'))
          AND (ip_discontinued_code IS NULL OR ip_discontinued_code NOT IN ('M', 'O'))
          AND ip_restriction_code = 'RTSN'
          AND (fp_product_type IS NULL OR fp_product_type NOT IN ('04', '07'))
          AND (fp_discontinued_code IS NULL OR fp_discontinued_code NOT IN ('M', 'O'))
          AND fp_restriction_code = 'RTSN'
    ), comp_master_desc AS (
        SELECT
            part_number,
            MAX(CASE WHEN part_description IS NOT NULL THEN 1 ELSE 0 END) AS has_comp_master_description,
            COUNT(DISTINCT CASE WHEN part_description IS NOT NULL THEN TRIM(part_description) END) AS distinct_comp_master_descriptions
        FROM XREF_ADMIN.XREF_COMP_PART_MASTER_EU_VW
        GROUP BY part_number
    )
    SELECT
        COUNT(*) AS distinct_active_comp2fish_ips_with_null_ip_description,
        SUM(CASE WHEN NVL(cmd.has_comp_master_description, 0) = 1 THEN 1 ELSE 0 END) AS ips_recovered_with_comp_master_description,
        SUM(CASE WHEN NVL(cmd.has_comp_master_description, 0) = 0 THEN 1 ELSE 0 END) AS ips_still_without_description,
        AVG(NVL(cmd.distinct_comp_master_descriptions, 0)) AS avg_distinct_comp_master_descriptions_per_ip
    FROM active_comp2fish_null_ip_desc x
    LEFT JOIN comp_master_desc cmd
      ON x.ip_number = cmd.part_number
""")

comp2fish_null_ip_description_recovery = add_rate(
    comp2fish_null_ip_description_recovery,
    "ips_recovered_with_comp_master_description",
    "distinct_active_comp2fish_ips_with_null_ip_description",
    "recovery_rate",
)
save_df(comp2fish_null_ip_description_recovery, "xref_comp2fish_null_ip_description_recovery.csv")
display(comp2fish_null_ip_description_recovery)

In [ ]:
comp2fish_null_ip_description_recovery_distribution = query_df("""
    WITH active_comp2fish_null_ip_desc AS (
        SELECT DISTINCT ip_number
        FROM XREF_ADMIN.XREF_COMP_VEND_FISH_EU_VW
        WHERE cross_type = 'COMP2FISH'
          AND ip_description IS NULL
          AND ip_number IS NOT NULL
          AND (ip_product_type IS NULL OR ip_product_type NOT IN ('04', '07'))
          AND (ip_discontinued_code IS NULL OR ip_discontinued_code NOT IN ('M', 'O'))
          AND ip_restriction_code = 'RTSN'
          AND (fp_product_type IS NULL OR fp_product_type NOT IN ('04', '07'))
          AND (fp_discontinued_code IS NULL OR fp_discontinued_code NOT IN ('M', 'O'))
          AND fp_restriction_code = 'RTSN'
    ), comp_master_desc AS (
        SELECT
            part_number,
            COUNT(DISTINCT CASE WHEN part_description IS NOT NULL THEN TRIM(part_description) END) AS distinct_comp_master_descriptions
        FROM XREF_ADMIN.XREF_COMP_PART_MASTER_EU_VW
        GROUP BY part_number
    )
    SELECT
        CASE
            WHEN NVL(cmd.distinct_comp_master_descriptions, 0) >= 5 THEN '5+'
            ELSE TO_CHAR(NVL(cmd.distinct_comp_master_descriptions, 0))
        END AS distinct_comp_master_description_bucket,
        CASE
            WHEN NVL(cmd.distinct_comp_master_descriptions, 0) >= 5 THEN 5
            ELSE NVL(cmd.distinct_comp_master_descriptions, 0)
        END AS bucket_sort,
        COUNT(*) AS distinct_ip_numbers
    FROM active_comp2fish_null_ip_desc x
    LEFT JOIN comp_master_desc cmd
      ON x.ip_number = cmd.part_number
    GROUP BY
        CASE
            WHEN NVL(cmd.distinct_comp_master_descriptions, 0) >= 5 THEN '5+'
            ELSE TO_CHAR(NVL(cmd.distinct_comp_master_descriptions, 0))
        END,
        CASE
            WHEN NVL(cmd.distinct_comp_master_descriptions, 0) >= 5 THEN 5
            ELSE NVL(cmd.distinct_comp_master_descriptions, 0)
        END
    ORDER BY bucket_sort
""")

save_df(comp2fish_null_ip_description_recovery_distribution, "xref_comp2fish_null_ip_description_recovery_distribution.csv")
display(comp2fish_null_ip_description_recovery_distribution)

In [ ]:
ax = comp2fish_null_ip_description_recovery_distribution.plot.bar(
    x="distinct_comp_master_description_bucket",
    y="distinct_ip_numbers",
    figsize=(8, 5),
    legend=False,
    color="tab:purple",
)
format_axis(ax, "COMP2FISH null IP_DESCRIPTION recovery from COMP master", "Distinct COMP master descriptions", "Distinct IP numbers")
plt.show()

## Next Notebook Context

The next notebooks should use these findings to build ground-truth datasets:

- External input pairs from active `COMP2FISH`: `IP_NUMBER` / external description → `FP_NUMBER`, `FP_LEAF_NODE_ID`.
- Internal input pairs from active `FISH2FISH`: `IP_NUMBER`, `IP_LEAF_NODE_ID` → `FP_NUMBER`, `FP_LEAF_NODE_ID`.
- Samples should be stratified by match difficulty, relationship type, MTS2/3, and language/description availability.

Vector-source work should compare multilingual combined documents vs language-specific vectors, document-field composition, multilingual embedding models, and reranking/fusion strategies.